In [1]:
import pandas as pd
import sqlite3
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.nn.utils.rnn import pack_padded_sequence
torch.manual_seed(112358)

In [2]:
conn=sqlite3.connect("../data/archive.sqlite3")
cursor=conn.cursor()
df=pd.read_sql_query("SELECT * FROM posts,post_tags where posts.thread_id=post_tags.thread_id",conn)

In [3]:
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
0,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,Vectors
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
...,...,...,...,...,...,...,...,...,...,...,...,...
1174634,345979,1556697,148231,1.393553e+09,2,0,"<div></div><a href=""http://www.artofproblemsol...",Kyiv Taras Shevchenko University Mechmat Compe...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities
1174635,345979,1556697,148231,1.393553e+09,2,0,"<div></div><a href=""http://www.artofproblemsol...",Kyiv Taras Shevchenko University Mechmat Compe...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities unsolved
1174636,345980,1556697,148231,1.398564e+09,2,0,<div></div>The following inequality is also tr...,The following inequality is also true.\nShow t...,0,https://artofproblemsolving.com/community/p155...,1556697,algebra
1174637,345980,1556697,148231,1.398564e+09,2,0,<div></div>The following inequality is also tr...,The following inequality is also true.\nShow t...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities


In [4]:
df=df[df.is_first_post==1]
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
0,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,Vectors
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
...,...,...,...,...,...,...,...,...,...,...,...,...
1174622,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,algebra unsolved
1174623,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,function
1174627,345977,1556697,46787,1.247241e+09,1,0,<div></div>Show that for all integers <span st...,"Show that for all integers $ n \ge 2$, $ \sqrt...",1,https://artofproblemsolving.com/community/p155...,1556697,algebra
1174628,345977,1556697,46787,1.247241e+09,1,0,<div></div>Show that for all integers <span st...,"Show that for all integers $ n \ge 2$, $ \sqrt...",1,https://artofproblemsolving.com/community/p155...,1556697,inequalities


In [5]:
df.memory_usage(deep=True)

Index               1346408
id                  1346408
thread_id           1346408
user_id             1346408
created_at          1346408
thanks_count        1346408
nothanks_count      1346408
raw_html          323690920
processed_html     73736543
is_first_post       1346408
source             16741166
thread_id           1346408
tag                10120919
dtype: int64

In [6]:
tag=df["tag"].value_counts()
tag

tag
geometry           19077
algebra            11493
number theory      11148
combinatorics      10196
inequalities        4823
                   ...  
sets of numbers        1
Prove that             1
airlines               1
cities                 1
pco                    1
Name: count, Length: 3617, dtype: int64

In [7]:
# we 'll take the first 4
tag=tag[tag>10000]
tag

tag
geometry         19077
algebra          11493
number theory    11148
combinatorics    10196
Name: count, dtype: int64

In [8]:
df=df[df.tag.isin(tag.index)]

In [9]:
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
5,4,24681313,167643,1.647280e+09,0,0,"<div></div>Find, with proof, all functions <im...","Find, with proof, all functions $f : R - \{0\}...",1,https://artofproblemsolving.com/community/p246...,24681313,algebra
...,...,...,...,...,...,...,...,...,...,...,...,...
1174607,345970,1556755,46787,1.247244e+09,2,0,"<div></div>Consider a convex solid <img src=""/...",Consider a convex solid $ K$ in space and two ...,1,https://artofproblemsolving.com/community/p155...,1556755,geometry
1174610,345971,1556694,46787,1.247241e+09,1,0,<div></div>Determine the number of integers <i...,Determine the number of integers $ n$ with $ 1...,1,https://artofproblemsolving.com/community/p155...,1556694,number theory
1174615,345973,1556701,46787,1.247241e+09,2,0,<div></div>In a convex quadrilateral <span sty...,"In a convex quadrilateral $ ABCD$, let $ E$ be...",1,https://artofproblemsolving.com/community/p155...,1556701,geometry
1174621,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,algebra


In [10]:
ds=df.loc[:, ~df.columns.duplicated()].groupby("thread_id",as_index=True).agg(
    text=("processed_html","first"),
    tag=("tag",list)
)
ds

,text,tag
thread_id,,
2,"Let $ABC$ be a triangle, and $M$ an interior p...",[geometry]
3,okay this one is from Prof. Mircea Lascu from ...,"[algebra, geometry]"
5,"If A,B are invertible and the set {Ak - Bk | k...",[algebra]
9,In a magic square $n \times n$ composed from t...,"[algebra, combinatorics]"
72,The lengths of the sides of a convex hexagon $...,[geometry]
...,...,...
36238654,"Let $a, b, c$ be the altitudes of triangle $A$...",[geometry]
36238689,Find all functions that satisfy the condition ...,[algebra]
36238706,On an $N \times N$ “chessboard” ($N \ge 3$) ea...,[combinatorics]


In [11]:
ds.memory_usage(deep=True)

Index      371648
text     19470208
tag       3729424
dtype: int64

In [12]:
sp = spm.SentencePieceProcessor(model_file="my_tokenizer.model")

In [13]:
def tokenize(text):
    return sp.encode(text,out_type=int)
X=ds["text"].apply(tokenize)
X

thread_id
2           [215, 3, 427, 7916, 81, 6, 332, 7929, 35, 3, 7...
3           [4808, 232, 149, 275, 29, 264, 387, 7924, 7937...
5           [320, 79, 7929, 7951, 101, 7886, 35, 9, 439, 4...
9           [622, 6, 7885, 471, 3, 7914, 8, 705, 45, 7916,...
72          [266, 2315, 31, 9, 927, 31, 6, 2166, 3058, 3, ...
                                  ...                        
36238654    [215, 3, 7912, 7929, 24, 7929, 18, 7916, 81, 9...
36238689    [1022, 170, 1892, 38, 1023, 9, 733, 98, 170, 5...
36238706    [1782, 22, 3, 7967, 8, 705, 147, 7916, 7909, 0...
36238733    [533, 29, 1689, 38, 156, 1611, 68, 2514, 256, ...
36238754    [622, 332, 3, 427, 48, 3, 311, 138, 217, 7958,...
Name: text, Length: 46456, dtype: object

In [14]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(ds["tag"])
print(mlb.classes_)

['algebra' 'combinatorics' 'geometry' 'number theory']


In [15]:
class ContestProblemDataset(Dataset):
    def __init__(self, X, Y):
        self.X = [torch.tensor(x, dtype=torch.long) for x in X]   
        self.Y = torch.tensor(Y, dtype=torch.float32)              

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def collate_fn(batch):
    xs, ys = zip(*batch)
    x_lens = torch.tensor([len(x) for x in xs])
    x_padded = pad_sequence(list(xs), batch_first=True, padding_value=8000)
    y_batch = torch.stack(ys)   
    return x_padded, y_batch, x_lens

In [16]:
BATCH_SIZE=16

X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.9,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_test,Y_test,test_size=0.5,random_state=42)

train_ds=ContestProblemDataset(X_train,Y_train)
val_ds=ContestProblemDataset(X_val,Y_val)
test_ds=ContestProblemDataset(X_test,Y_test)

train_loader=DataLoader(train_ds,BATCH_SIZE,shuffle=True,collate_fn=collate_fn)
val_loader=DataLoader(val_ds,BATCH_SIZE,shuffle=False,collate_fn=collate_fn)
test_loader=DataLoader(test_ds,BATCH_SIZE,shuffle=False,collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 291
Validation batches: 1307
Test batches: 1307


In [17]:
train_ds.X

[tensor([6758,   44,    9, 2527,    9,  585,    3, 5079,  782, 7922,  180, 3141,
         7944,  133,  786,  735,   42, 7944, 7937,    3,  336, 7913, 7976, 7967,
           80,  693, 4367,   92,   13,  537, 7913, 7976]),
 tensor([1022,  170,    9, 2398,    3, 7957, 7936, 7944,  109,   31,    6, 1465,
           43,  298,   45, 7916,  144,  566,  758, 7943, 1748, 2071,  230,   38,
            3, 7957, 7936, 7944, 7935,    8,  134,  145,  212,   34,  186, 7944,
          931,    8,  298,  336, 7957, 1047, 7976,  835,  468,    3,    8, 1101,
          166, 7964,  217, 2929]),
 tensor([ 280,  354,    7, 2103,  100,   80,    9,  855, 3288, 7937,  215,  305,
         5546, 5002,    3,    8,   34, 7927, 7917, 2493, 7911, 4682,   35,    3,
            8,   34, 7927, 7917, 3529, 7911, 5489,   81,  850,  159,   44, 4093,
         7917, 7929,  144,    9, 3341, 4812,   35, 6821, 1519,  380,  529, 7937,
          240, 7055,    3,   12,  570,   35,    3,   12,  699,  153, 1287, 3213,
         1453, 

In [18]:
class CateogoryLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout, padding_idx):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim,padding_idx)
        self.LSTM=nn.LSTM(embedding_dim,hidden_dim,num_layers=n_layers,bidirectional=True,dropout=dropout,batch_first=True)
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(2*hidden_dim,output_dim)
    def forward(self, text_batch, x_lens):
        embedded = self.embedding(text_batch)
        
        packed = pack_padded_sequence(embedded, x_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (hidden, cell) = self.LSTM(packed)
        
        # hidden shape: [num_layers * num_directions, batch, hidden_dim]
        bi_hidden = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        
        dropped_out = self.dropout(bi_hidden)
        linear_out = self.fc(dropped_out)
        
        return linear_out

EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 4
N_LAYERS = 2
DROPOUT = 0.5
PADDING_IDX = 8000

# Instantiate the model
model = CateogoryLSTM(8001, 
                        EMBEDDING_DIM, 
                        HIDDEN_DIM, 
                        OUTPUT_DIM, 
                        N_LAYERS, 
                        DROPOUT, 
                        PADDING_IDX)

print(model)



CateogoryLSTM(
  (embedding): Embedding(8001, 100, padding_idx=8000)
  (LSTM): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=4, bias=True)
)


In [21]:
# --- Training Hyperparameters ---
LEARNING_RATE = 0.001
N_EPOCHS = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=LEARNING_RATE)

model=model.to(device)
criteron=criterion.to(device)


def jaccard_accuracy(y_true, logits, threshold=0.5):
    probs = torch.sigmoid(logits)
    y_pred_bin = (probs > threshold).float()
    
    intersection = (y_true * y_pred_bin).sum(dim=1)
    union = ((y_true + y_pred_bin) > 0).float().sum(dim=1)
    jaccard = torch.where(union > 0, intersection / union, torch.ones_like(union))
    return jaccard.mean()

Training on device: cpu


In [ ]:


for epoch in range(N_EPOCHS):
    
    train_loss = 0.0
    train_acc = 0.0
    
    # --- Training Phase ---
    model.train() # Set model to training mode
    
    for inputs, labels,x_len in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} [Train]"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 1. Zero gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        predictions = model(inputs,x_len)
        
        # 3. Calculate loss and accuracy
        loss = criterion(predictions, labels)
        acc = jaccard_accuracy(labels,predictions)
        
        # 4. Backward pass
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        train_loss += loss.item()
        train_acc += acc.item()



Training on device: cpu


Epoch 5/5 [Train]: 100%|██████████| 291/291 [06:19<00:00,  1.30s/it]


In [19]:
# torch.save(model.state_dict(),"weight.model")
model.load_state_dict(torch.load("weight.model"))

model


CateogoryLSTM(
  (embedding): Embedding(8001, 100, padding_idx=8000)
  (LSTM): LSTM(100, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=4, bias=True)
)

In [ ]:
model.eval()
text="""Let $\mathbb{N}$ denote the set of positive integers. A function $f\colon\mathbb{N}\to\mathbb{N}$ is said to be bonza if
\[
f(a)~~\text{divides}~~b^a-f(b)^{f(a)}
\]for all positive integers $a$ and $b$.

Determine the smallest real constant $c$ such that $f(n)\leqslant cn$ for all bonza functions $f$ and all positive integers $n$."""
tokens = sp.encode(text, out_type=int)
inp = torch.tensor(tokens).unsqueeze(0).to(device)
x_lens = torch.tensor([len(tokens)])  

with torch.no_grad():
    prediction = model(inp, x_lens)
    probs = torch.sigmoid(prediction)
    predicted_labels = (probs > 0.5).int()
    predicted_indices = torch.nonzero(predicted_labels.squeeze()).flatten().tolist()
    predicted_names = [mlb.classes_[i] for i in predicted_indices]
    print(predicted_names)  # ví dụ: ['algebra', 'geometry']

['number theory']


<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\huyho\AppData\Local\Temp\ipykernel_2196\2942980154.py:2: SyntaxWarning: invalid escape sequence '\m'
  text="""Let $\mathbb{N}$ denote the set of positive integers. A function $f\colon\mathbb{N}\to\mathbb{N}$ is said to be bonza if
